# Notebook 00 — Установка и настройка проекта

**Запускай этот ноутбук один раз** при первом открытии Colab.

Что делает этот ноутбук автоматически:
1. Проверяет GPU
2. Подключает Google Drive
3. Клонирует проект с GitHub на Drive
4. Устанавливает все зависимости
5. Проверяет что всё импортируется без ошибок

---
> ⚠️ **Перед запуском:** убедись что выбран GPU:  
> `Среда выполнения → Сменить тип среды → GPU (T4)`
>
> Затем нажми: `Среда выполнения → Выполнить все` (Ctrl+F9)

## Ячейка 1 из 5 — Проверка GPU

In [ ]:
import torch

print('=' * 50)
print('  ПРОВЕРКА ОКРУЖЕНИЯ')
print('=' * 50)
print(f'  PyTorch:  {torch.__version__}')
print(f'  CUDA:     {torch.cuda.is_available()}')

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU:      {name}')
    print(f'  Память:   {mem:.1f} GB')
    print('=' * 50)
    print('  ✓ GPU готов к работе')
else:
    print('=' * 50)
    print('  ✗ GPU не найден!')
    print('  Перейди: Среда выполнения → Сменить тип среды → GPU')
    raise RuntimeError('Нужен GPU. Смени тип среды и перезапусти.')

## Ячейка 2 из 5 — Подключение Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Корневая папка проекта на Drive
PROJECT_DIR = '/content/drive/MyDrive/diploma'

print(f'\n✓ Google Drive подключён')
print(f'  Проект будет в: {PROJECT_DIR}')

## Ячейка 3 из 5 — Клонирование проекта с GitHub

In [ ]:
import os
import subprocess
import shutil

GITHUB_USERNAME = 'ravenMVA'
REPO_NAME       = 'diploma-autonomous-driving'

REPO_URL    = f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'
PROJECT_DIR = '/content/drive/MyDrive/diploma'
TEMP_DIR    = '/content/diploma_tmp'

# Клонируем во временную папку
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)

print(f'Клонируем {REPO_URL} ...')
result = subprocess.run(
    ['git', 'clone', REPO_URL, TEMP_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('ОШИБКА:', result.stderr)
    raise RuntimeError('Не удалось клонировать репозиторий.')

# Копируем файлы проекта в PROJECT_DIR, не трогая папку data/
os.makedirs(PROJECT_DIR, exist_ok=True)
for item in os.listdir(TEMP_DIR):
    if item == 'data':
        continue  # data/ не трогаем — там лежит датасет
    src = os.path.join(TEMP_DIR, item)
    dst = os.path.join(PROJECT_DIR, item)
    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

shutil.rmtree(TEMP_DIR)

print('\nСтруктура проекта:')
for root, dirs, files in os.walk(PROJECT_DIR):
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != 'data']
    level = root.replace(PROJECT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

print('\n✓ Проект готов')

## Ячейка 4 из 5 — Установка зависимостей

In [ ]:
import subprocess, sys

req_path = os.path.join(PROJECT_DIR, 'requirements.txt')

print('Устанавливаем зависимости...')
print('(занимает ~1-2 минуты)\n')

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', req_path],
    capture_output=True, text=True
)

if result.returncode != 0:
    print('Предупреждения при установке:')
    print(result.stderr[:500])

# Создаём рабочие папки
for folder in ['data', 'models', 'outputs']:
    path = os.path.join(PROJECT_DIR, folder)
    os.makedirs(path, exist_ok=True)
    print(f'✓ Папка: {path}')

print('\n✓ Зависимости установлены')

## Ячейка 5 из 5 — Проверка импортов

In [ ]:
import sys

PROJECT_DIR = '/content/drive/MyDrive/diploma'
src_path    = os.path.join(PROJECT_DIR, 'src')

if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Проверяем каждый модуль
checks = []

try:
    import torch
    checks.append(('torch', True, torch.__version__))
except Exception as e:
    checks.append(('torch', False, str(e)))

try:
    import cv2
    checks.append(('opencv-python', True, cv2.__version__))
except Exception as e:
    checks.append(('opencv-python', False, str(e)))

try:
    from ultralytics import YOLO
    checks.append(('ultralytics (YOLOv8)', True, 'OK'))
except Exception as e:
    checks.append(('ultralytics (YOLOv8)', False, str(e)))

try:
    from model import PilotNet
    m = PilotNet()
    checks.append(('model.py (PilotNet)', True, 'OK'))
except Exception as e:
    checks.append(('model.py (PilotNet)', False, str(e)))

try:
    from dataset import get_dataloaders
    checks.append(('dataset.py', True, 'OK'))
except Exception as e:
    checks.append(('dataset.py', False, str(e)))

try:
    from train import train
    checks.append(('train.py', True, 'OK'))
except Exception as e:
    checks.append(('train.py', False, str(e)))

try:
    from evaluate import evaluate
    checks.append(('evaluate.py', True, 'OK'))
except Exception as e:
    checks.append(('evaluate.py', False, str(e)))

# Выводим результаты
print('=' * 55)
print('  ПРОВЕРКА КОМПОНЕНТОВ')
print('=' * 55)
all_ok = True
for name, ok, info in checks:
    status = '✓' if ok else '✗'
    print(f'  {status}  {name:<28} {info}')
    if not ok:
        all_ok = False
print('=' * 55)

if all_ok:
    print('\n  ✓ Всё готово! Можешь открывать notebook 01.')
    print(f'  Проект: {PROJECT_DIR}')
else:
    print('\n  ✗ Есть ошибки. Проверь сообщения выше.')

---
## Что дальше?

1. Скачай датасет с Kaggle: `ronamgir/udacity-car-dataset-simulator-challenge`
2. Загрузи архив в `diploma/data/` на Google Drive, переименуй в `dataset.zip`
3. Открывай ноутбуки по порядку:

| Ноутбук | Что делает |
|---------|------------|
| `01_setup_and_data.ipynb` | Загрузка и анализ датасета |
| `02_preprocessing.ipynb` | Предобработка и аугментация |
| `03_model_training.ipynb` | Обучение PilotNet |
| `04_evaluation.ipynb` | Оценка качества |
| `05_yolo_detection.ipynb` | Детекция объектов (YOLO) |